# 01_CFT Treatment Interactions Database

Builds the statewide, non-overlapping atomic treatment interactions layer from
the flagged CFT dataset produced by `00_CFT_Ingest.ipynb`.

Flagged records (`ac_flag | ac_flag_abs`) are excluded before processing.
No temporal filter is applied — all retained treatments are preserved.
Downstream workflows (e.g., TEALOM) apply their own AOI clip and temporal
filters against the output.

**Input**
- `data/spatial/mod/CFTv2_CO_Flagged.gpkg`

**Output**
- `data/spatial/mod/CFTv2_CO_Interactions.gpkg` — atomic treatment zones


In [ ]:
import sys, os
from pathlib import Path

import geopandas as gpd

# --- Paths
code_dir = Path.cwd().parent          # .../treatment_interactions/code/
proj_dir = code_dir.parent            # .../treatment_interactions/
box_dir = code_dir.parents[2]
print(box_dir)
sys.path.insert(0, str(code_dir))

from cft_interactions.interactions import build_treatment_interactions
from cft_interactions.helpers import (
    events_to_rows,
    filter_by_activity,
    filter_by_year_range,
    filter_complete,
)

FLAGGED_FP = proj_dir / 'data/spatial/mod/CFTv2_CO_AllTreatments_Flagged.gpkg'
INTERACTIONS_FP = proj_dir / 'data/spatial/mod/CFTv2_CO_AllTreatments_Interactions.gpkg'
PROJ_CRS   = 26913   # NAD83 UTM Zone 13N

m2_to_acres = 0.000247105 # conversion (multiplication) of m2 to acres
min_acres = (30*30)*m2_to_acres # approximate size of one 30m pixel
print(f"Minimum acre threshold: {min_acres}")

print(f'Project dir: {proj_dir}')

## 01_Load and filter flagged dataset

In [ ]:
cft_flagged = gpd.read_file(FLAGGED_FP)
print(f'Loaded: {len(cft_flagged):,} features')

# Exclude flagged records — both residual and absolute flags
cft_clean = cft_flagged[~(cft_flagged['ac_flag'] | cft_flagged['ac_flag_abs'])].copy()
n_excl = len(cft_flagged) - len(cft_clean)
print(f'Excluded (flagged): {n_excl:,}')
print(f'Clean input:        {len(cft_clean):,} features')

## 02_Build atomic interactions database

In [ ]:
# --- Clip to a county boundary for testing
fp = os.path.join(box_dir, 'data/boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp')
counties = gpd.read_file(fp).to_crs(cft_clean.crs)
boulder = counties[counties['NAME'] == 'Boulder']
# --- Extract treatments
boco_trts = gpd.overlay(cft_clean, boulder, how='intersection')
print(f"Boulder County: {len(boco_trts):,} features")

In [ ]:
interactions = build_treatment_interactions(
    boco_trts,
    source_id_col='OBJECTID',
    snap_tolerance=0.5,
    min_acres=0.11,
    n_workers=3
)

In [ ]:
# --- Save the file out
out_fp = os.path.join(proj_dir, 'data/spatial/mod/BoCo_Treatment_Interactions.gpkg')
interactions.to_file(out_fp, index=False)
print(f"Interactions saved to {out_fp}")

## 03_Summary

In [ ]:
print(f'Total atomic zones:    {len(interactions):,}')
print(f'Complete (thin+burn):  {interactions["COMPLETE"].sum():,}')
print(f'\nComplete type breakdown:')
print(interactions['COMPLETE_TYPE'].value_counts(dropna=False))
print(f'\nYear range: {interactions["FIRST_TRT_YEAR"].min()}–{interactions["LAST_TRT_YEAR"].max()}')
print(f'Agencies (unique): {len(set("|".join(interactions["AGENCIES"].dropna()).split("|")))}')
print(f'\nAtom size (acres):\n{interactions["ACRES_GIS"].describe()}')

In [ ]:
# Spot-check: verify SOURCE_IDs link back to OBJECTID in original CFT
import json
sample = interactions.iloc[0]
src_ids = json.loads(sample['SOURCE_IDS'])
events  = json.loads(sample['EVENTS'])
print(f'ATOM_ID {sample["ATOM_ID"]}  ({sample["ACRES_GIS"]:.2f} ac)')
print(f'SOURCE_IDS: {src_ids}')
print(f'Events:')
for e in events:
    print(f'  {e}')

## 04_Helper demonstrations

In [ ]:
# Explode to one row per event for tabular analysis
event_rows = events_to_rows(interactions)
print(f'Event rows: {len(event_rows):,}  (should equal N_EVENTS sum = {interactions["N_EVENTS"].sum():,})')
print(event_rows.groupby('ACTIVITY')['ACRES_GIS'].sum().sort_values(ascending=False))

In [ ]:
# Filter examples (TEALOM-style, applied ad-hoc)
recent = filter_by_year_range(interactions, 2021, 2024)
print(f'Treatments with any event 2021–2024: {len(recent):,}')

complete = filter_complete(interactions)
print(f'Complete treatment zones: {len(complete):,}')

mechanical = filter_by_activity(interactions, ['Mechanical'])
print(f'Zones with Mechanical thinning: {len(mechanical):,}')

## 05_Save output

In [ ]:
INTERACTIONS_FP.parent.mkdir(parents=True, exist_ok=True)
interactions.to_file(INTERACTIONS_FP)
print(f'Saved {len(interactions):,} atomic zones → {INTERACTIONS_FP}')
print(f'Columns: {list(interactions.columns)}')